# Smart Lender - Loan Prediction
## Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('../Dataset/loan_prediction.csv')
df.head()

In [ ]:
# 1. Count Plots for Categorical Features
plt.figure(figsize=(15, 10))

cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Credit_History']
for i, col in enumerate(cat_cols, 1):
    plt.subplot(3, 3, i)
    sns.countplot(x=col, data=df, palette='viridis')
    plt.title(f'Count Plot of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Distribution Plots for Numerical Features
plt.figure(figsize=(15, 5))

num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
for i, col in enumerate(num_cols, 1):
    plt.subplot(1, 3, i)
    sns.histplot(df[col].dropna(), kde=True, color='blue', bins=30)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Bar Charts: Relationship with Target (Loan_Status)
plt.figure(figsize=(15, 10))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(3, 3, i)
    sns.countplot(x=col, hue='Loan_Status', data=df, palette='Set2')
    plt.title(f'{col} vs Loan_Status')
plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
# 1. Handling Missing Values
# Numerical values: Fill with Mean
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].mean())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mean())
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mean())

# Categorical values: Fill with Mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

print("Missing values after filling:")
print(df.isnull().sum())


In [ ]:
# 2. Encoding Categorical Variables
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Columns to encode
cols_to_encode = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for col in cols_to_encode:
    df[col] = le.fit_transform(df[col].astype(str))

# Drop Loan_ID as it's not a useful feature for prediction
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

print("Data after encoding and dropping Loan_ID:")
print(df.head())


## Model Building

In [ ]:
# 1. Splitting Data and Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
with open('../Flask/scale1.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../Flask/scale1(1).pkl', 'wb') as f:
    pickle.dump(scaler, f)


In [ ]:
# 2. Model Training and Evaluation
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

best_model = None

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    
    print(f"{name}:")
    print(f"  Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
    print(f"  Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}\n")
    
    if name == 'XGBoost':
        best_model = model

# Save the best model (XGBoost) as rdf.pkl to match structure
with open('../Flask/rdf.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("XGBoost model saved to Flask/rdf.pkl")
